# Data Inventory of CHU Data in PSWS Database

In [1]:
# import time
# import pandas as pd
# import requests
# from bs4 import BeautifulSoup

# BASE_URL = (
#     "https://pswsnetwork.eng.ua.edu/observations/observation_list/"
#     "?centerFrequency=3.330&centerFrequency=7.850&centerFrequency=14.670"
#     "&station=&startDate__gte=&endDate__lte="
#     "&latitude_min=&latitude_max=&longitude_min=&longitude_max="
# )

# headers = {
#     "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36"
# }


# def build_inventory_with_links(max_pages=30000):
#     """Scrapes the HTML table and parses out the actual download links for each observation."""
#     all_rows = []
#     current_page = 1

#     while current_page <= max_pages:
#         print(f"Indexing metadata from page {current_page}...")
#         url = f"{BASE_URL}&page={current_page}"

#         try:
#             response = requests.get(url, headers=headers, timeout=12)
#             if response.status_code != 200:
#                 break

#             soup = BeautifulSoup(response.text, "html.parser")
#             table = soup.find("table", class_="obsTable")

#             if not table:
#                 print("Table not found. Ending.")
#                 break

#             # Find all data rows inside the table body
#             tbody = table.find("tbody")
#             if not tbody:
#                 break

#             for tr in tbody.find_all("tr"):
#                 tds = tr.find_all("td")
#                 if len(tds) < 9:  # Safeguard for unexpected row shapes
#                     continue

#                 # Safely extract text from the standard table cells
#                 data_rate = tds[0].get_text(strip=True)
#                 freqs = tds[1].get_text(strip=True)
#                 station = tds[2].get_text(strip=True)
#                 instrument = tds[3].get_text(strip=True)
#                 size_mb = tds[4].get_text(strip=True)
#                 file_text = tds[5].get_text(strip=True)
#                 start_utc = tds[7].get_text(strip=True)
#                 end_utc = tds[8].get_text(strip=True)

#                 # --- THE MAGIC STEP ---
#                 # Look inside the File/Observation cell (tds[5]) for the actual link
#                 file_link_tag = tds[5].find("a")
#                 download_url = ""
#                 obs_id = ""

#                 if file_link_tag and "href" in file_link_tag.attrs:
#                     relative_path = file_link_tag["href"]  # e.g., "/observations/select_download_range/167207/"
#                     download_url = f"https://pswsnetwork.eng.ua.edu{relative_path}"

#                     # Clean extract the unique Observation ID from the path strings
#                     path_parts = [p for p in relative_path.split("/") if p]
#                     if path_parts:
#                         obs_id = path_parts[-1]

#                 # Append everything cleanly into our master data collector
#                 all_rows.append(
#                     {
#                         "Observation_ID": obs_id,
#                         "Station": station,
#                         "Instrument": instrument,
#                         "Center_Frequencies": freqs,
#                         "Size_MB": size_mb,
#                         "File_Label": file_text,
#                         "Start_UTC": start_utc,
#                         "End_UTC": end_utc,
#                         "Direct_Download_URL": download_url,
#                     }
#                 )

#             # Pagination boundary check
#             pagination_ul = soup.find("ul", class_="pagination")
#             if pagination_ul:
#                 available_pages = [
#                     int(a["href"].split("page=")[-1])
#                     for a in pagination_ul.find_all("a")
#                     if "page=" in a.get("href", "")
#                 ]
#                 if available_pages and (current_page + 1) > max(available_pages):
#                     print("Reached final page listing.")
#                     break

#             current_page += 1
#             time.sleep(1.0)

#         except Exception as e:
#             print(f"Error on page {current_page}: {e}")
#             break

#     return pd.DataFrame(all_rows)


# # Run it!
# df_inventory = build_inventory_with_links(max_pages=2000)
# print(f"\nInventory built with {len(df_inventory)} items!")

# # Let's inspect our new dataframe columns, including the hidden download links!
# print(df_inventory[["Station", "Observation_ID", "Direct_Download_URL"]].head())

# df_inventory.to_csv('CHU Data in PSWS Database.csv')

In [ ]:
import os
import time
import pandas as pd
import requests
from bs4 import BeautifulSoup

BASE_URL = (
    "https://pswsnetwork.eng.ua.edu/observations/observation_list/"
    "?centerFrequency=3.330&centerFrequency=7.850&centerFrequency=14.670"
    "&station=&startDate__gte=&endDate__lte="
    "&latitude_min=&latitude_max=&longitude_min=&longitude_max="
)

headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36"
}

OUTPUT_FILE = "CHU_Data_in_PSWS_Database.csv"


def build_inventory_with_links(max_pages=30000, max_retries=5):
    """Scrapes the HTML table, handles timeouts with exponential backoff,

    and picks up where it left off using a checkpoint CSV file.
    """
    all_rows = []
    current_page = 1

    # --- CHECKPOINT RESUME LOGIC ---
    # If the file already exists, determine the last successfully processed page
    if os.path.exists(OUTPUT_FILE):
        try:
            existing_df = pd.read_csv(OUTPUT_FILE)
            if not existing_df.empty and "Scraped_Page" in existing_df.columns:
                last_page = existing_df["Scraped_Page"].max()
                current_page = int(last_page) + 1
                print(
                    f"Found existing backup. Resuming from page {current_page}..."
                )
                # Load the existing rows into memory so we can return the full dataframe at the end
                all_rows = existing_df.to_dict("records")
        except Exception as e:
            print(
                f"Could not parse existing CSV file ({e}). Starting from scratch."
            )

    while current_page <= max_pages:
        print(f"Indexing metadata from page {current_page}...")
        url = f"{BASE_URL}&page={current_page}"

        page_rows = []
        retry_count = 0
        backoff_delay = 5  # Start with a 5-second delay if a timeout occurs
        success = False

        # --- RETRY & TIMEOUT LOOP ---
        while retry_count < max_retries and not success:
            try:
                # Kept your 12s timeout, which is a solid window
                response = requests.get(url, headers=headers, timeout=12)

                if response.status_code == 429:
                    print(
                        f"Rate limited (429). Backing off for {backoff_delay}s..."
                    )
                    time.sleep(backoff_delay)
                    retry_count += 1
                    backoff_delay *= 2
                    continue

                if response.status_code != 200:
                    print(
                        f"Server returned status code {response.status_code} on page {current_page}. Exiting."
                    )
                    # Break out of retry loop, success stays False, which breaks main loop
                    break

                # If code 200, we successfully connected!
                soup = BeautifulSoup(response.text, "html.parser")
                success = True

            except (
                requests.exceptions.Timeout,
                requests.exceptions.ConnectionError,
            ) as te:
                retry_count += 1
                print(
                    f"Timeout/Connection error on page {current_page} (Attempt {retry_count}/{max_retries}): {te}"
                )
                if retry_count < max_retries:
                    print(f"Waiting {backoff_delay} seconds before retrying...")
                    time.sleep(backoff_delay)
                    backoff_delay *= 2  # Exponentially increase wait time
                else:
                    print(
                        f"Max retries reached for page {current_page}. Critical failure."
                    )

            except requests.exceptions.RequestException as re:
                print(f"General request exception on page {current_page}: {re}")
                break

        # If we failed to get the page after all retries, terminate safely without losing data
        if not success:
            print(
                f"Stopping execution at page {current_page}. Returning data collected so far."
            )
            break

        # --- PARSING LOGIC ---
        table = soup.find("table", class_="obsTable")
        if not table:
            print(f"Table not found on page {current_page}. Ending.")
            break

        tbody = table.find("tbody")
        if not tbody:
            print(f"Table body empty on page {current_page}. Ending.")
            break

        for tr in tbody.find_all("tr"):
            tds = tr.find_all("td")
            if len(tds) < 9:
                continue

            data_rate = tds[0].get_text(strip=True)
            freqs = tds[1].get_text(strip=True)
            station = tds[2].get_text(strip=True)
            instrument = tds[3].get_text(strip=True)
            size_mb = tds[4].get_text(strip=True)
            file_text = tds[5].get_text(strip=True)
            start_utc = tds[7].get_text(strip=True)
            end_utc = tds[8].get_text(strip=True)

            file_link_tag = tds[5].find("a")
            download_url = ""
            obs_id = ""

            if file_link_tag and "href" in file_link_tag.attrs:
                relative_path = file_link_tag["href"]
                download_url = f"https://pswsnetwork.eng.ua.edu{relative_path}"

                path_parts = [p for p in relative_path.split("/") if p]
                if path_parts:
                    obs_id = path_parts[-1]

            row_data = {
                "Observation_ID": obs_id,
                "Station": station,
                "Instrument": instrument,
                "Center_Frequencies": freqs,
                "Size_MB": size_mb,
                "File_Label": file_text,
                "Start_UTC": start_utc,
                "End_UTC": end_utc,
                "Direct_Download_URL": download_url,
                "Scraped_Page": current_page,  # Track page number for tracking checkpoints
            }
            page_rows.append(row_data)
            all_rows.append(row_data)

        # --- INCREMENTAL SAVE LOGIC ---
        # Append just this page's results directly to the disk file
        if page_rows:
            page_df = pd.DataFrame(page_rows)
            # If it's page 1 (and no file existed), write headers. Otherwise, just append rows.
            if current_page == 1 and not os.path.exists(OUTPUT_FILE):
                page_df.to_csv(OUTPUT_FILE, index=False)
            else:
                page_df.to_csv(
                    OUTPUT_FILE, mode="a", header=not os.path.exists(OUTPUT_FILE), index=False
                )

        # Pagination boundary check
        pagination_ul = soup.find("ul", class_="pagination")
        if pagination_ul:
            available_pages = [
                int(a["href"].split("page=")[-1])
                for a in pagination_ul.find_all("a")
                if "page=" in a.get("href", "")
            ]
            if available_pages and (current_page + 1) > max(available_pages):
                print("Reached final page listing.")
                break

        current_page += 1
        time.sleep(1.0)  # Politeness delay

    return pd.DataFrame(all_rows)


# Run it!
df_inventory = build_inventory_with_links(max_pages=2000)
print(f"\nInventory built/loaded with {len(df_inventory)} items!")
print(df_inventory[["Station", "Observation_ID", "Direct_Download_URL"]].head())

Found existing backup. Resuming from page 263...
Indexing metadata from page 263...
Indexing metadata from page 264...
Indexing metadata from page 265...
Indexing metadata from page 266...
Indexing metadata from page 267...
Indexing metadata from page 268...
Indexing metadata from page 269...
Indexing metadata from page 270...
Indexing metadata from page 271...
Indexing metadata from page 272...
Indexing metadata from page 273...
Indexing metadata from page 274...
Indexing metadata from page 275...
Indexing metadata from page 276...
Indexing metadata from page 277...
Indexing metadata from page 278...
Indexing metadata from page 279...
Indexing metadata from page 280...
Indexing metadata from page 281...
Indexing metadata from page 282...
Indexing metadata from page 283...
Indexing metadata from page 284...
Indexing metadata from page 285...
Indexing metadata from page 286...
Indexing metadata from page 287...
Indexing metadata from page 288...
Indexing metadata from page 289...
Indexi

In [ ]:
# To download a specific row later:
file_response = requests.get(df_inventory.loc[0, "Direct_Download_URL"], headers=headers)
with open(f"{df_inventory.loc[0, 'File_Label']}.zip", "wb") as f:
    f.write(file_response.content)

## Gantt

In [ ]:
import pandas as pd
import plotly.express as px

# 1. Ensure datetimes are properly parsed
df_inventory["Start_UTC"] = pd.to_datetime(df_inventory["Start_UTC"])
df_inventory["End_UTC"] = pd.to_datetime(df_inventory["End_UTC"])

# 2. Clean and explode frequencies to isolate CHU channels
df_faceted = df_inventory.copy()
df_faceted["Center_Frequencies"] = (
    df_faceted["Center_Frequencies"]
    .str.replace(" MHz", "", regex=False)
    .str.split(", ")
)
df_faceted = df_faceted.explode("Center_Frequencies")

# Filter down strictly to CHU channels
chu_frequencies = ["3.330", "7.850", "14.670"]
df_faceted = df_faceted[df_faceted["Center_Frequencies"].isin(chu_frequencies)]
df_faceted["Frequency_Header"] = df_faceted["Center_Frequencies"] + " MHz"

# --- THE CRITICAL STEP ---
# 3. Convert Size_MB from text string to a float number for continuous mapping
df_faceted["Size_MB"] = pd.to_numeric(df_faceted["Size_MB"], errors="coerce")

# 4. Sort the data
df_faceted = df_faceted.sort_values(by=["Frequency_Header", "Station", "Start_UTC"])

# 5. Generate the Faceted Gantt Chart with Continuous Color
fig = px.timeline(
    df_faceted,
    x_start="Start_UTC",
    x_end="End_UTC",
    y="Station",
    color="Size_MB",               # Continuous mapping triggered by numeric data
    color_continuous_scale="Cividis_r",  # "Viridis", "Plasma", or "Cividis" work beautifully here
    facet_row="Frequency_Header",
    category_orders={"Frequency_Header": ["3.330 MHz", "7.850 MHz", "14.670 MHz"]},
    title="PSWS Network Observations Inventory by CHU Frequency",
    hover_data={
        "Instrument": True,
        "Size_MB": ":.2f",         # Format hover text to two decimal places
        "Start_UTC": "|%Y-%m-%d %H:%M",
        "End_UTC": "|%Y-%m-%d %H:%M",
    },
)

# 6. Layout adjustments
fig.update_layout(
    height=800,
    xaxis_title="Time (UTC)",
    yaxis_title="Station",
    coloraxis_colorbar_title="File Size (MB)",  # Set title of the continuous colorbar
    title_x=0.5,
)

# Clean up the automatic row labels on the right side of the subplots
fig.for_each_annotation(lambda a: a.update(text=a.text.split("=")[-1]))

fig.show()

In [ ]:
fig.write_html('CHU Gantt.html')

## Map

In [ ]:
pip install maidenhead

### Scrape locations:

In [ ]:
import time
import pandas as pd
import requests
from bs4 import BeautifulSoup

STATIONS_URL = "https://pswsnetwork.eng.ua.edu/stations/stations/"
headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36"
}


def scrape_station_grid_lookup(target_stations_list):
    """Scrapes the PSWS stations database to find Grid squares for our specific inventory stations."""
    # Convert list to a set for fast, O(1) matching lookups
    missing_stations = set(target_stations_list)
    grid_lookup_dict = {}

    current_page = 1
    print(f"Beginning Grid lookup for {len(missing_stations)} unique stations...")

    while missing_stations:
        print(f"Searching Stations Directory page {current_page}...")
        url = f"{STATIONS_URL}?page={current_page}"

        try:
            response = requests.get(url, headers=headers, timeout=12)

            # If we run past the final page of the directory, stop
            if response.status_code != 200:
                print(f"Reached directory boundary at page {current_page}.")
                break

            # Parse html tables found on the current directory page
            tables = pd.read_html(response.text)
            if not tables:
                print("No tables found. Ending lookup loop.")
                break

            # The station index table has columns: ['ID', 'User', 'Nickname', 'Grid', ...]
            df_page = tables[0]

            # Iterate through the rows on this directory page
            for _, row in df_page.iterrows():
                nickname = str(row.get("Nickname", "")).strip()
                grid = str(row.get("Grid", "")).strip()

                # If this station matches one we need, store its grid square
                if nickname in missing_stations:
                    # Filter out placeholders or blanks if any exist
                    if grid and grid != "—" and grid != "nan":
                        grid_lookup_dict[nickname] = grid
                        missing_stations.remove(nickname)

            # Look ahead in the pagination list to see if another page exists
            soup = BeautifulSoup(response.text, "html.parser")
            pagination_ul = soup.find("ul", class_="pagination")

            if pagination_ul:
                available_pages = [
                    int(a["href"].split("page=")[-1])
                    for a in pagination_ul.find_all("a")
                    if "page=" in a.get("href", "")
                ]
                if available_pages and (current_page + 1) > max(available_pages):
                    print("Scraped all available directory pages.")
                    break
            else:
                break

            current_page += 1
            time.sleep(1.0)  # Standard safety pause between pagination pages

        except Exception as e:
            print(f"Error reading directory page {current_page}: {e}")
            break

    print(
        f"\nLookup complete! Found grids for {len(grid_lookup_dict)} stations."
    )
    if missing_stations:
        print(f"Could not locate grid configurations for: {list(missing_stations)}")

    return grid_lookup_dict


# --- HOW TO RUN IT WITH YOUR EXISTING DATAFRAME ---
# 1. Pull the unique station strings directly out of your master inventory dataframe
unique_inventory_stations = df_inventory["Station"].unique().tolist()

# 2. Run the dynamic scraper to fetch the grid codes
station_grid_lookup = scrape_station_grid_lookup(unique_inventory_stations)

# 3. View your freshly generated custom translation dictionary
print("\nGenerated Dictionary:")
print(station_grid_lookup)

In [ ]:
import maidenhead as mh
import pandas as pd
import plotly.express as px

# --- SCRIPT SETUP: Sample data lookup for the stations in your inventory ---
# The server lists Grid squares on the stations tab. 
# Here's a quick dictionary mapping common PSWS stations to their Grid locations.
# station_grid_lookup = {
#     "pa0rwt Grape-1 DRF": "JO21tr",
#     "N4RVE": "EM74tr",
#     "W1EUJ": "FN42im",
#     "KFS_NW": "CM87uu",
#     "AC0G_ND": "EM38ww",
#     "K9TRV RX888-1": "EN51aa",
#     "WA2TP-2": "FN20lb",
# }


def grid_to_coordinates(station_name):
    """Looks up a station's ham radio grid locator and converts it to Lat/Lon."""
    grid = station_grid_lookup.get(station_name, None)
    if grid:
        try:
            # Returns (latitude, longitude) center point of the grid square
            lat, lon = mh.to_location(grid)
            return pd.Series([lat, lon])
        except Exception:
            return pd.Series([None, None])
    return pd.Series([None, None])


# 1. Map the Grid coordinates into your current Dataframe
# (Assuming df_inventory is the clean dataframe you generated earlier)
df_map = df_inventory.copy()
df_map[["Latitude", "Longitude"]] = df_map["Station"].apply(grid_to_coordinates)

# Drop any stations we didn't have grid definitions for in the dictionary helper
df_map = df_map.dropna(subset=["Latitude", "Longitude"])

# 2. Group the data so each station becomes a single unique dot on the map
df_stations = (
    df_map.groupby(["Station", "Latitude", "Longitude", "Instrument"])
    .agg(
        Total_Observations=("Observation_ID", "count"),
        Frequencies_Monitored=("Center_Frequencies", lambda x: ", ".join(set(x))),
    )
    .reset_index()
)

# 3. Generate the Interactive Geographic Map
fig = px.scatter_map(
    df_stations,
    lat="Latitude",
    lon="Longitude",
    color="Instrument",  # Distinct color for every physical hardware configuration
    size="Total_Observations",  # Size of the dot scales with dataset inventory size
    hover_name="Station",
    hover_data={
        "Frequencies_Monitored": True,
        "Total_Observations": True,
        "Latitude": ":.3f",
        "Longitude": ":.3f",
    },
    title="Geographic Footprint of CHU Monitoring Stations",
    zoom=2,
)

# 4. Apply a clean OpenStreetMap background style layout
fig.update_layout(
    margin={"r": 0, "t": 40, "l": 0, "b": 0},
    title_x=0.5,
    map_style="open-street-map",  # Crisp, fully public global map background
)

fig.show()

In [ ]:
import maidenhead as mh
import pandas as pd
import plotly.express as px

# 1. Ensure timestamps are proper datetime objects
df_map_timeline = df_inventory.copy()
df_map_timeline["Start_UTC"] = pd.to_datetime(df_map_timeline["Start_UTC"])

# 2. Add a Year-Month column to act as our animation steps
df_map_timeline["Year_Month"] = df_map_timeline["Start_UTC"].dt.to_period("M").astype(str)

# 3. Apply your scraped grid lookup dictionary to get coordinates
# (Using the station_grid_lookup dictionary generated by your previous step)
def grid_to_coordinates(station_name):
    grid = station_grid_lookup.get(station_name, None)
    if grid:
        try:
            lat, lon = mh.to_location(grid)
            return pd.Series([lat, lon])
        except Exception:
            return pd.Series([None, None])
    return pd.Series([None, None])

df_map_timeline[["Latitude", "Longitude"]] = df_map_timeline["Station"].apply(grid_to_coordinates)
df_map_timeline = df_map_timeline.dropna(subset=["Latitude", "Longitude"])

# 4. Group the data by Station AND Month so we can see operational changes over time
df_animated = (
    df_map_timeline.groupby(["Year_Month", "Station", "Latitude", "Longitude", "Instrument"])
    .agg(Observations_This_Month=("Observation_ID", "count"))
    .reset_index()
)

# 5. Crucial Step: Sort by date so the animation frames play in chronological order
df_animated = df_animated.sort_values("Year_Month")

# 6. Build the Animated Map
fig = px.scatter_map(
    df_animated,
    lat="Latitude",
    lon="Longitude",
    color="Instrument",
    size="Observations_This_Month",
    hover_name="Station",
    hover_data={"Observations_This_Month": True, "Year_Month": False},
    
    # --- ANIMATION CONFIGURATION ---
    animation_frame="Year_Month",  # This creates the draggable timeline bar
    animation_group="Station",     # Tells Plotly to track the same station across frames
    
    title="Growth of CHU Monitoring Network Over Time",
    zoom=2,
)

# 7. Layout Adjustments
fig.update_layout(
    margin={"r": 0, "t": 40, "l": 0, "b": 0},
    title_x=0.5,
    map_style="open-street-map",
    height=650
)

# Optional: Make the animation transition speed slightly faster/smoother (in milliseconds)
# fig.layout.updatemenus[0].buttons[0].args[1]["frame"]["duration"] = 800

fig.show()
fig.write_html('CHU Over Time.html')